In [1]:
import csv
import pandas as pd
import os
import numpy as np
from datetime import datetime

In [2]:
base_dir = os.getcwd()
idb_dir = os.path.join(base_dir, "IDB/Update_2024")
mafap_dir = os.path.join(base_dir, "MAFAP/Update_2024")
oecd_dir = os.path.join(base_dir, "OECD/Update_2024")

In [3]:
# OECD Exchange Rate Data
er_dir = os.path.join(oecd_dir,"other_data")
oecd_ex = pd.read_excel(os.path.join(er_dir,"XratesMon2024.xlsx"), skiprows=1)
oecd_ex = oecd_ex.drop(['Data we publish online', 'Unnamed: 1', 'Source','Comments'], axis=1)
oecd_ex = pd.melt(oecd_ex,id_vars=['Country','Currency code','Currency name'],var_name="Year",value_name="Exchange_Rate")
oecd_ex = oecd_ex.replace("EU","EUR")

oecd_ex.rename(columns={'Country':'Country_Code'}, inplace=True)

oecd_ex = oecd_ex[['Country_Code','Year','Exchange_Rate']]
oecd_ex.head()

,Country_Code,Year,Exchange_Rate
0,ARG,1986,0.000094
1,AUS,1986,1.495996
2,AUT,1986,15.268078
3,BEL,1986,NaN
4,BGR,1986,0.000940


In [4]:
# IDB Exchange Rate Data 
countries = ['BRAZIL','CANADA','EUROPEAN UNION','COSTA RICA','COLOMBIA','CHILE','MEXICO','UNITED STATES','ARGENTINA']
idb_ex = pd.read_csv(os.path.join(idb_dir,"./Data/ExchangeRates.csv"))

idb_ex.set_index('country', inplace=True)
idb_ex = pd.DataFrame(idb_ex.unstack().reset_index())
idb_ex.rename(columns={'level_0':'Year', 'country':'Country_Label', 0:'Exchange_Rate'}, inplace=True)
idb_ex['Country_Label'] = idb_ex['Country_Label'].str.strip()
idb_ex = idb_ex[~idb_ex['Country_Label'].isin(countries)]
idb_ex = idb_ex[~idb_ex['Country_Label'].isin(['Canada','OECD total'])]

idb_ex.Country_Label = idb_ex.Country_Label.str.lower().str.title()
idb_ex.Country_Label = idb_ex.Country_Label.str.replace('And','and')

idb_ex.Year = idb_ex.Year.astype(int)
idb_ex = idb_ex[idb_ex['Year']>=2006]
idb_ex = idb_ex.reset_index(drop=True)
idb_ex = idb_ex[['Country_Label','Year','Exchange_Rate']]
idb_ex = idb_ex.sort_values(['Country_Label','Year'])
idb_ex.head()


,Country_Label,Year,Exchange_Rate
0,Bahamas,2006,NaN
19,Bahamas,2007,NaN
38,Bahamas,2008,NaN
57,Bahamas,2009,NaN
76,Bahamas,2010,1.0


In [5]:
mafap_ex = pd.read_csv(os.path.join(mafap_dir,"Exchange_Rate_2024.csv"), nrows=24)
mafap_ex = mafap_ex.drop(['Series Name','Series Code'],axis=1)
mafap_ex.rename(columns={'Country Code':'Country_Code', 'Country Name':'Country_Label'}, inplace=True)
mafap_ex = pd.melt(mafap_ex,id_vars=['Country_Label','Country_Code'],var_name='Year', value_name='Exchange_Rate')
mafap_ex['Year'] = mafap_ex['Year'].apply(lambda x: x.split(' ')[0]).astype(int)
mafap_ex = mafap_ex.sort_values(['Country_Code','Year'])
mafap_ex = mafap_ex[mafap_ex.Country_Label.notnull()]
mafap_ex.Country_Label = mafap_ex.Country_Label.str.replace('Kyrgyz Republic','Kyrgyzstan')
mafap_ex.head()

,Country_Label,Country_Code,Year,Exchange_Rate
17,Armenia,ARM,2005,457.686941
41,Armenia,ARM,2006,416.040370
65,Armenia,ARM,2007,342.079116
89,Armenia,ARM,2008,305.969400
113,Armenia,ARM,2009,363.283286


In [6]:
# eth_exch = pd.read_excel(os.path.join(mafap_dir, "Ethiopia_Exchange_Rate.xlsx"), nrows=2)
# # eth_exch = eth_exch.drop(['Country Name'],axis=1)
# eth_exch.rename(columns={'Country Code':'Country_Code', 'Country Name':'Country_Label'}, inplace=True)
# eth_exch = pd.melt(eth_exch,id_vars=['Country_Label','Country_Code'],var_name='Year', value_name='Exchange_Rate')
# eth_exch['Year'] = eth_exch['Year'].apply(lambda x: x.split(' ')[0]).astype(int)
# eth_exch.head()

In [7]:
# droping WDI data for Ethiopia. Using exchange rate from MAFAP database 
# mafap_ex = mafap_ex[mafap_ex['Country_Code']!='ETH']
# mafap_ex = mafap_ex.append(eth_exch)
# mafap_ex.shape

In [8]:
output_dir = os.path.join(oecd_dir,"output")
oecd_data = pd.read_csv(os.path.join(output_dir,"OECD_Payment_Data.csv"))
oecd_data = pd.melt(oecd_data, id_vars=['Country_Label','Country_Code','Commodity_Label', 'Commodity_Code','Commodity_Type',
                                        'Year'], var_name='Category', value_name='Value_LCU')
oecd_data.Year = oecd_data.Year.astype(int)

oecd_data = oecd_data.merge(oecd_ex, how='left')
oecd_data['Value_USD'] = oecd_data['Value_LCU']/oecd_data['Exchange_Rate']
oecd_data['Source'] ='OECD'

oecd_data.Country_Label = oecd_data.Country_Label.str.lower().str.title()

oecd_data = oecd_data[['Source','Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type',
                                        'Year','Category','Value_LCU','Value_USD']]
oecd_data.head()

,Source,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,Category,Value_LCU,Value_USD
0,OECD,Argentina,ARG,Beef and veal,BF,Yes,2006,A2,NaN,NaN
1,OECD,Argentina,ARG,Beef and veal,BF,Yes,2007,A2,NaN,NaN
2,OECD,Argentina,ARG,Beef and veal,BF,Yes,2008,A2,NaN,NaN
3,OECD,Argentina,ARG,Beef and veal,BF,Yes,2009,A2,NaN,NaN
4,OECD,Argentina,ARG,Beef and veal,BF,Yes,2010,A2,NaN,NaN


In [9]:
output_dir = os.path.join(mafap_dir,"Output")
mafap_data = pd.read_csv(os.path.join(output_dir,"MAFAP_Payment_Data.csv"))

mafap_data = pd.melt(mafap_data, id_vars=['Country_Label','Country_Code','Commodity_Label', 'Commodity_Code','Commodity_Type',
                                        'Year'], var_name='Category', value_name='Value_LCU')

mafap_data = mafap_data[mafap_data.Country_Code!='ZWE']

mafap_data = mafap_data.merge(mafap_ex, how='left')
mafap_data['Value_USD'] = mafap_data['Value_LCU']/mafap_data['Exchange_Rate']
mafap_data['Source'] ='FAO'

mafap_data.Commodity_Type = np.where(mafap_data.Commodity_Label=='Unallocated', np.nan, mafap_data.Commodity_Type)

mafap_data = mafap_data[['Source','Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type',
                            'Year','Category','Value_LCU','Value_USD']]

mafap_data = mafap_data[~(mafap_data.Value_USD.isnull())]
mafap_data.Commodity_Code = mafap_data.Commodity_Code.astype(int)

mafap_data.head()

,Source,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,Category,Value_LCU,Value_USD
48,FAO,Azerbaijan,AZE,Non-allocated crops,9991,No,2018,A2,1584000.0,9.317556e+05
56,FAO,Azerbaijan,AZE,Seed cotton,328,No,2016,A2,8872000.0,5.559867e+06
57,FAO,Azerbaijan,AZE,Seed cotton,328,No,2017,A2,20719000.0,1.203785e+07
58,FAO,Azerbaijan,AZE,Seed cotton,328,No,2018,A2,29591000.0,1.740630e+07
59,FAO,Azerbaijan,AZE,"Tobacco, unmanufactured",826,No,2017,A2,95000.0,5.519550e+04


In [10]:
output_dir = os.path.join(idb_dir,"output")
idb_data = pd.read_csv(os.path.join(output_dir,"IDB_Payment_Data.csv"))

idb_data = pd.melt(idb_data, id_vars=['Country_Label','Country_Code','Commodity_Label', 'Commodity_Code','Commodity_Type',
                                        'Year'], var_name='Category', value_name='Value_LCU')

idb_data = idb_data.merge(idb_ex, how='left')
idb_data['Value_USD'] = idb_data['Value_LCU']/idb_data['Exchange_Rate']
idb_data['Source'] = 'IDB'

idb_data = idb_data[['Source','Country_Label','Country_Code','Commodity_Label','Commodity_Code','Commodity_Type',
                            'Year','Category','Value_LCU','Value_USD']]

idb_data.head()

,Source,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,Category,Value_LCU,Value_USD
0,IDB,Bahamas,BHS,Poultry meat,31,Yes,2010,A2,NaN,NaN
1,IDB,Bahamas,BHS,Poultry meat,31,Yes,2011,A2,NaN,NaN
2,IDB,Bahamas,BHS,Poultry meat,31,Yes,2012,A2,NaN,NaN
3,IDB,Bahamas,BHS,Poultry meat,31,Yes,2013,A2,NaN,NaN
4,IDB,Bahamas,BHS,Poultry meat,31,Yes,2014,A2,NaN,NaN


In [11]:
consolidated = pd.concat([oecd_data, idb_data, mafap_data])
print(consolidated.shape)
consolidated = consolidated[consolidated.Value_LCU!=0]
consolidated = consolidated[consolidated.Value_LCU.notnull()]
consolidated = consolidated[~(consolidated.Category.isin(['B1','B2','B3']))]

(73342, 10)


In [12]:
len(idb_data.Country_Code.unique())

19

In [13]:
# consolidated.to_csv('Consolidated_Payment_Data.csv', index=False)

In [14]:
# consolidated_data = pd.read_csv('Consolidated_Payment_Data_Old.csv')
# consolidated_data['Country_Label'] = np.where(consolidated_data['Country_Label']=='E28', 
#                                               'European Union', consolidated_data['Country_Label'])
# consolidated_data['Country_Code'] = np.where(consolidated_data['Country_Code']=='E28', 
#                                               'EUR', consolidated_data['Country_Code'])


# consolidated_data.shape

In [15]:
additional_data = pd.read_excel('Additional_country_data.xlsx')
additional_data = additional_data.groupby(['Source','Country_Label','Country_Code','Commodity_Label',
                                        'Commodity_Code','Commodity_Type','Year','Category']).sum()[['Value_LCU','Value_USD']].reset_index()

additional_data.Commodity_Type = np.nan
additional_data.head()

,Source,Country_Label,Country_Code,Commodity_Label,Commodity_Code,Commodity_Type,Year,Category,Value_LCU,Value_USD
0,Alam et al 2020,Bangladesh,BGD,Unallocated,9999,NaN,2017,B,86427000000,1.106620e+09
1,Djanibekov & Petrick 2018,Uzbekistan,UZB,Unallocated,9999,NaN,2016,B,1590013800000,4.941000e+08
2,IFPRI,Pakistan,PAK,Unallocated,9999,NaN,2014,B,525700000,5.110830e+06
3,Kassim et al 2018 & Kurdi et al 2020,Egypt,EGY,Unallocated,9999,NaN,2018,B,2000000000,1.140000e+08
4,Michael et al 2018,Nigeria,NGA,Unallocated,9999,NaN,2014,B,76000000000,4.793361e+08


In [16]:
consolidated_add = consolidated.append(additional_data)
consolidated_add.shape

C:\Users\AMAMUN\AppData\Local\Temp\ipykernel_18212\3074449293.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  consolidated_add = consolidated.append(additional_data)


(11713, 10)

In [17]:
consolidated_add.to_csv(os.path.join(base_dir, './Output/Consolidated_PaymentData_2024/Consolidated_Payment_Data_2024.csv'), index=False, encoding="utf-8-sig")

In [18]:
commodities =consolidated_add[['Source','Commodity_Label','Commodity_Code']].drop_duplicates()
commodities.to_csv('Commodity_List.csv', index=False)

In [19]:
consolidated_list = consolidated_add[['Source','Commodity_Label',
                                      'Commodity_Code']].drop_duplicates().reset_index(drop=True)
print(consolidated_list.shape)
consolidated_list['Exist']=1
consolidated_list.to_csv(os.path.join(base_dir, './Output/Consolidated_PaymentData_2024/Commodity_List_2024.csv'), index=False, encoding="utf-8-sig")

(223, 3)


In [20]:
# consolidate_compare = pd.read_csv('Consolidated_Payment_Data_Compare.csv')
# consolidate_compare = consolidate_compare[['Source','Commodity_Label',
#                                            'Commodity_Code']].drop_duplicates().reset_index(drop=True)
# print(consolidate_compare.shape)

# consolidate_compare['Exist_old']=1
# consolidate_compare.to_csv('Commodity_List_Old.csv', index=False)

In [21]:
# consolidate_diff = consolidated_add.merge(consolidate_compare, how='left')
# consolidate_diff.to_csv('consolidate_diff.csv', index=False)